# Verify Top Gainers and Movers

This notebook verifies the calculation of top inflationary and deflationary categories.

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('cleaned_inflation_with_calculations.csv')
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

print(f"Dataset loaded: {len(df)} rows")
print(f"Date range: {df['time'].min()} to {df['time'].max()}")

Dataset loaded: 176 rows
Date range: 2011-01-01 00:00:00 to 2025-08-01 00:00:00


In [2]:
# Define exclusions
meta_cols = ['time', 'Year', 'Month', 'Month_Name', 'CPI_YoY', 'CPI_MoM']

EXCLUDE_EXACT = {
    'Consumer Price Index',
    'Consumer Food Price Index',
    'Core Bottom Up', 
    'Core Index', 
    'CPI (ExVeggies)', 
    'Core Ex TnC Index',
    'Core Core Exc Index', 
    'Custom Index 1', 
    'Custom Index 2', 
    'Exclusion Index',
    'RBI Core Index', 
    'Protein Index',
    'Consumer Price Index: exclude Food & Beverages and Fuel & Light',
    'Consumer Price Index: exclude Food and Beverages',
    'Consumer Price Index: Food and Beverages',
    'Consumer Price Index: Miscellaneous',
    'Consumer Price Index: Clothing and Footwear'
}

EXCLUDE_KEYWORDS = [
    'Custom Index', 'Core Index', 'Core Bottom Up', 'Core Ex TnC',
    'Core Core Exc', 'Exclusion Index', 'RBI Core Index', 'Protein Index',
    '(ExVeggies)', 'exclude Food'
]

def is_excluded(colname: str) -> bool:
    if colname in EXCLUDE_EXACT:
        return True
    for kw in EXCLUDE_KEYWORDS:
        if kw.lower() in colname.lower():
            return True
    return False

# Get valid CPI columns
candidates = [c for c in df.columns if c not in meta_cols]
cpi_cols = [c for c in candidates if not is_excluded(c)]

print(f"\nTotal CPI component columns: {len(cpi_cols)}")
print("\nColumns included:")
for col in cpi_cols:
    print(f"  - {col}")


Total CPI component columns: 24

Columns included:
  - Consumer Price Index: Food and Beverages: Cereals and Products
  - Consumer Price Index: Food and Beverages: Meat and Fish
  - Consumer Price Index: Food and Beverages: Egg
  - Consumer Price Index: Food and Beverages: Milk and Milk Product
  - Consumer Price Index: Food and Beverages: Oils and Fats
  - Consumer Price Index: Food and Beverages: Fruits
  - Consumer Price Index: Food and Beverages: Vegetables
  - Consumer Price Index: Food and Beverages: Pulses and Products
  - Consumer Price Index: Food and Beverages: Sugar and Confectionery
  - Consumer Price Index: Food and Beverages: Spices
  - Consumer Price Index: Food and Beverages: Non-alcholic Beverages
  - Consumer Price Index: Food and Beverages: Prepared Meals, Snacks, Sweets, etc
  - Consumer Price Index: Pan, Tobacco and Intoxicants
  - Consumer Price Index: Clothing and Footwear: Clothing
  - Consumer Price Index: Clothing and Footwear: Footwear
  - Consumer Price Ind

In [3]:
# Get latest and previous month
latest_idx = df.index[-1]
prev_idx = df.index[-2]

latest_date = df.loc[latest_idx, 'time'].strftime('%B %Y')
prev_date = df.loc[prev_idx, 'time'].strftime('%B %Y')

print(f"Previous month: {prev_date}")
print(f"Latest month: {latest_date}")

Previous month: July 2025
Latest month: August 2025


In [4]:
# Calculate month-on-month changes
rows = []
for col in cpi_cols:
    prev_v = df.loc[prev_idx, col]
    cur_v = df.loc[latest_idx, col]
    if pd.notna(prev_v) and pd.notna(cur_v) and prev_v != 0:
        change = ((cur_v - prev_v) / prev_v) * 100
        if abs(change) > 0.01:
            rows.append({
                'Category': col,
                'Previous Value': prev_v,
                'Current Value': cur_v,
                'MoM Change (%)': change
            })

changes_df = pd.DataFrame(rows).sort_values('MoM Change (%)', ascending=False)

print(f"\nTotal categories with changes: {len(changes_df)}")
print(f"\nAll changes sorted (showing all):")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
display(changes_df)


Total categories with changes: 23

All changes sorted (showing all):


,Category,Previous Value,Current Value,MoM Change (%)
6,Consumer Price Index: Food and Beverages: Vege...,212.1,219.2,3.347478
4,Consumer Price Index: Food and Beverages: Oils...,192.1,195.8,1.926080
22,Consumer Price Index: Miscellaneous: Personal ...,230.7,233.1,1.040312
8,Consumer Price Index: Food and Beverages: Suga...,135.3,136.3,0.739098
15,Consumer Price Index: Housing,185.7,186.7,0.538503
17,Consumer Price Index: Miscellaneous: Household...,184.9,185.5,0.324500
21,Consumer Price Index: Miscellaneous: Education,195.0,195.5,0.256410
0,Consumer Price Index: Food and Beverages: Cere...,197.1,197.6,0.253678
18,Consumer Price Index: Miscellaneous: Health,203.5,204.0,0.245700
11,Consumer Price Index: Food and Beverages: Prep...,211.0,211.5,0.236967


In [5]:
# Get top 5 increases and decreases
top_5_increases = changes_df[changes_df['MoM Change (%)'] > 0].head(5).copy()
top_5_decreases = changes_df[changes_df['MoM Change (%)'] < 0].tail(5).copy()

print("\n" + "="*80)
print("TOP 5 INFLATIONARY CHANGES (INCREASES)")
print("="*80)
display(top_5_increases)

print("\n" + "="*80)
print("TOP 5 DEFLATIONARY CHANGES (DECREASES)")
print("="*80)
display(top_5_decreases)


TOP 5 INFLATIONARY CHANGES (INCREASES)


,Category,Previous Value,Current Value,MoM Change (%)
6,Consumer Price Index: Food and Beverages: Vege...,212.1,219.2,3.347478
4,Consumer Price Index: Food and Beverages: Oils...,192.1,195.8,1.926080
22,Consumer Price Index: Miscellaneous: Personal ...,230.7,233.1,1.040312
8,Consumer Price Index: Food and Beverages: Suga...,135.3,136.3,0.739098
15,Consumer Price Index: Housing,185.7,186.7,0.538503



TOP 5 DEFLATIONARY CHANGES (DECREASES)


,Category,Previous Value,Current Value,MoM Change (%)
9,Consumer Price Index: Food and Beverages: Spices,220.8,220.7,-0.045290
7,Consumer Price Index: Food and Beverages: Puls...,184.3,183.5,-0.434075
1,Consumer Price Index: Food and Beverages: Meat...,229.3,226.6,-1.177497
2,Consumer Price Index: Food and Beverages: Egg,198.6,194.9,-1.863041


In [6]:
# Summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Total categories analyzed: {len(changes_df)}")
print(f"Categories with increases: {len(changes_df[changes_df['MoM Change (%)'] > 0])}")
print(f"Categories with decreases: {len(changes_df[changes_df['MoM Change (%)'] < 0])}")
print(f"\nLargest increase: {changes_df['MoM Change (%)'].max():.2f}%")
print(f"Largest decrease: {changes_df['MoM Change (%)'].min():.2f}%")
print(f"Average change: {changes_df['MoM Change (%)'].mean():.2f}%")
print(f"Median change: {changes_df['MoM Change (%)'].median():.2f}%")


SUMMARY STATISTICS
Total categories analyzed: 23
Categories with increases: 19
Categories with decreases: 4

Largest increase: 3.35%
Largest decrease: -1.86%
Average change: 0.29%
Median change: 0.21%
